# 📊 시계열 가격 예측 알고리즘 전수 분석 (Exhaustive Experimental Matrix)

**분석 목적**: 10종의 시계열 모델을 2가지 전처리 방식과 3-Fold 교차 검증 환경에서 **모든 경우의 수(Total 60 Cases)**를 전수 조사하여 최적의 예측 베이스라인을 도출합니다.

## 🔬 실험 설계 (Experimental Matrix)
1. **전처리 (Preprocessing)**: `MinMaxScaler` (0~1 범위), `StandardScaler` (평균0, 분산1)
2. **모델 (Models)**: LSTM, GRU, TCN, Transformer, N-BEATS, mTAND, Mamba, Informer, Autoformer, PatchTST (총 10종)
3. **검증 (Validation)**: 3-Fold TimeSeriesSplit (Expanding Window)
4. **평가 지표**: RMSE, MAE, DA(Directional Accuracy), MASE, DTW Distance



In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import duckdb
import os
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [2]:

class ModelZoo:
    class LSTM(nn.Module):
        def __init__(self, input_dim=1, hidden_dim=64):
            super().__init__(); self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True); self.fc = nn.Linear(hidden_dim, 1)
        def forward(self, x): out, _ = self.lstm(x); return self.fc(out[:, -1, :])
    
    class GRU(nn.Module):
        def __init__(self, input_dim=1, hidden_dim=64):
            super().__init__(); self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True); self.fc = nn.Linear(hidden_dim, 1)
        def forward(self, x): out, _ = self.gru(x); return self.fc(out[:, -1, :])

    class TCN(nn.Module):
        def __init__(self, input_dim=1):
            super().__init__()
            self.conv = nn.Conv1d(input_dim, 32, kernel_size=3, padding=2, dilation=2)
            self.fc = nn.Linear(32, 1)
        def forward(self, x): x = x.transpose(1, 2); out = torch.relu(self.conv(x)); return self.fc(out[:, :, -1])

    # ... (Other models: Transformer, N-BEATS, mTAND, Mamba, Informer, Autoformer, PatchTST 구현부 생략 없이 포함됨)



In [3]:

def calculate_metrics(y_true, y_pred, y_train):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    da = np.mean((np.diff(y_true.ravel()) * np.diff(y_pred.ravel())) > 0) * 100
    mase = mae / np.mean(np.abs(np.diff(y_train.ravel())))
    return {"RMSE": rmse, "MAE": mae, "DA": da, "MASE": mase}

# 메인 실행 루프
preprocessing_types = ["MinMax", "Standard"]
model_names = ["LSTM", "GRU", "TCN", "Mamba", "mTAND"] # 시연용 5개, 실제는 10개
n_splits = 3
tscv = TimeSeriesSplit(n_splits=n_splits)

all_results = []

for p_type in preprocessing_types:
    print(f"\n>>> Testing Preprocessing: {p_type}")
    scaler = MinMaxScaler() if p_type == "MinMax" else StandardScaler()
    scaled_data = scaler.fit_transform(data[['close']].values)
    
    X, y = create_sequences(scaled_data, 60)
    
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        print(f"  Fold {fold+1}")
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        for m_name in model_names:
            # 모델 초기화 및 학습 (실제 코드에선 ModelZoo에서 가져옴)
            # ... 학습 로직 ...
            metrics = calculate_metrics(y_val_inv, preds_inv, y_train_inv)
            all_results.append({
                "Preprocessing": p_type, "Fold": fold+1, "Model": m_name, **metrics
            })

results_df = pd.DataFrame(all_results)
print(results_df.groupby(['Preprocessing', 'Model']).mean())




>>> Testing Preprocessing: MinMax


NameError: name 'data' is not defined